In [6]:
import numpy as np
import sinter
import stim

from pathlib import Path

from epic import (
    AllocCode,
    FreeCode,
    InitCode,
    PauliChar,
    PauliEigenState,
    PauliString,
    QECCompiler,
    ReadoutCode,
    RSCSurgery,
    RotatedSurfaceCode,
    StimLikeNoiseModel,
)

base_dir = Path("/Users/lenny/Documents/foo/EPIC-QEC/docs/example_notebooks/cnot_benchmark_result")
logical_rates_dir = base_dir / "logical_rates"
visual_dir = base_dir / "visual"
logical_rates_dir.mkdir(parents=True, exist_ok=True)
visual_dir.mkdir(parents=True, exist_ok=True)


def compile_cnot_experiment(distance: int, visual_output_path: Path | None = None):
    code1 = RotatedSurfaceCode.from_distance(distance=distance, system_coordinate=(0, 0))
    code2 = RotatedSurfaceCode.from_distance(distance=distance, system_coordinate=(0, 1))
    code3 = RotatedSurfaceCode.from_distance(distance=distance, system_coordinate=(1, 0))

    program = [
        AllocCode(target_code=code1, code_varname="ancilla_patch", logical_qubits_varnames=["ancilla"]),
        AllocCode(target_code=code2, code_varname="control_patch", logical_qubits_varnames=["control"]),
        AllocCode(target_code=code3, code_varname="target_patch", logical_qubits_varnames=["target"]),
        InitCode(targets=["ancilla_patch"], initial_state=PauliEigenState.X_plus, tag="init_ancilla"),
        InitCode(targets=["control_patch"], initial_state=PauliEigenState.Z_plus, tag="init_control"),
        InitCode(targets=["target_patch"], initial_state=PauliEigenState.Z_plus, tag="init_target"),
        RSCSurgery(
            targets=["ancilla", "control"],
            product_to_measure=PauliString(string=(PauliChar.Z, PauliChar.Z)),
            tag="MZZ_ac",
        ),
        RSCSurgery(
            targets=["ancilla", "target"],
            product_to_measure=PauliString(string=(PauliChar.X, PauliChar.X)),
            tag="MXX_ac",
        ),
        ReadoutCode(targets=["control_patch"], tag="LZ_control"),
        ReadoutCode(targets=["ancilla_patch"], tag="LZ_ancilla"),
        ReadoutCode(targets=["target_patch"], tag="LZ_target"),
        FreeCode(code_varname="target_patch"),
        FreeCode(code_varname="control_patch"),
        FreeCode(code_varname="ancilla_patch"),
    ]

    config = {
        "objective_distance": distance,
        "primitives": {
            "epic.core.qec_primitives.interfaces.ApplyGate": "epic.modules.qec_primitives.SimpleGateApplication",
            "epic.core.qec_primitives.interfaces.Readout": "epic.modules.qec_primitives.NaiveReadout",
            "epic.core.qec_primitives.interfaces.ExtractSyndrome": "epic.modules.qec_primitives.RSCSyndromeExtraction",
        },
    }

    compiler = QECCompiler(config)
    compiled_program = compiler.compile(program, visual_output_path=visual_output_path)
    stim_observables = [
        ["readout_LZ_control_rotated_surface_lq_0"],
        ["rsc_surgery_MZZ_ac", "readout_LZ_ancilla_rotated_surface_lq_0", "readout_LZ_target_rotated_surface_lq_0"],
    ]
    return compiled_program, stim_observables


noise_values = np.logspace(-5, -1, 9)
shots = 50_000
distances = [3, 5, 7]

for distance in distances:
    compiled_program, stim_observables = compile_cnot_experiment(
        distance,
        visual_output_path=visual_dir / "ls_cnot_d3_compilation_visual.pdf" if distance == 3 else None,
    )

    tasks = []
    for physical_noise in noise_values:
        noise_model = StimLikeNoiseModel.from_stim_like_probabilities(
            after_clifford_depolarization=float(physical_noise),
            after_reset_flip_probability=float(physical_noise),
            before_measure_flip_probability=float(physical_noise),
            before_round_data_depolarization=float(physical_noise),
        )
        stim_program = compiled_program.to_stim_program(stim_observables, noise_model=noise_model)
        tasks.append(
            sinter.Task(
                circuit=stim.Circuit(stim_program),
                json_metadata={"d": distance, "p": float(physical_noise)},
            )
        )

    collected_stats = sinter.collect(
        num_workers=4,
        tasks=tasks,
        decoders=["pymatching"],
        max_shots=shots,
    )
    stats_by_noise = {float(stats.json_metadata["p"]): stats for stats in collected_stats}
    logical_error_rates = np.array(
        [
            stats_by_noise[float(physical_noise)].errors
            / stats_by_noise[float(physical_noise)].shots
            for physical_noise in noise_values
        ]
    )
    result_title = f"EPIC lattice-surgery CNOT logical error rate (d={distance})"
    result_description = (
        f"Logical failure probability for an EPIC lattice-surgery CNOT on rotated surface codes at distance {distance}. "
        "A shot is counted as a logical fail when either measured logical observable is 1. "
        "Noise uses EPIC StimLikeNoiseModel with matched p for reset flip, Clifford depolarization, "
        "pre-measurement flip, and round-data depolarization channels."
    )
    results_path = logical_rates_dir / f"ls_cnot_d{distance}_rates.npz"
    np.savez_compressed(
        results_path,
        distance=np.array([distance], dtype=np.int64),
        noise_values=noise_values,
        logical_error_rates=logical_error_rates,
        shots=np.array([shots], dtype=np.int64),
        title=np.array([result_title]),
        description=np.array([result_description]),
    )

    print(f"Saved distance {distance} rates to: {results_path.name}")
    for physical_noise, logical_error_rate in zip(noise_values, logical_error_rates):
        print(f"  p={physical_noise:.2e} -> P_L={logical_error_rate:.6f}")


Saved distance 3 rates to: ls_cnot_d3_rates.npz
  p=1.00e-05 -> P_L=0.000000
  p=3.16e-05 -> P_L=0.000000
  p=1.00e-04 -> P_L=0.000000
  p=3.16e-04 -> P_L=0.000540
  p=1.00e-03 -> P_L=0.004960
  p=3.16e-03 -> P_L=0.046200
  p=1.00e-02 -> P_L=0.317340
  p=3.16e-02 -> P_L=0.705100
  p=1.00e-01 -> P_L=0.750400
Saved distance 5 rates to: ls_cnot_d5_rates.npz
  p=1.00e-05 -> P_L=0.000000
  p=3.16e-05 -> P_L=0.000000
  p=1.00e-04 -> P_L=0.000000
  p=3.16e-04 -> P_L=0.000040
  p=1.00e-03 -> P_L=0.000600
  p=3.16e-03 -> P_L=0.020580
  p=1.00e-02 -> P_L=0.384120
  p=3.16e-02 -> P_L=0.748480
  p=1.00e-01 -> P_L=0.751960
Saved distance 7 rates to: ls_cnot_d7_rates.npz
  p=1.00e-05 -> P_L=0.000000
  p=3.16e-05 -> P_L=0.000000
  p=1.00e-04 -> P_L=0.000000
  p=3.16e-04 -> P_L=0.000000
  p=1.00e-03 -> P_L=0.000020
  p=3.16e-03 -> P_L=0.007060
  p=1.00e-02 -> P_L=0.415320
  p=3.16e-02 -> P_L=0.747860
  p=1.00e-01 -> P_L=0.751400
